In [2]:
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

OUTPUT_PATH = Path('data') / '链接.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

LIST_SOURCES = {
    '京城大师赛': [
        'https://space.bilibili.com/19106800/lists/6648988?type=season',
        'https://space.bilibili.com/19106800/lists/7520632?type=season',
        'https://space.bilibili.com/19106800/lists/8209896?type=season',
    ],
    '粤小': [
        'https://space.bilibili.com/19106800/lists/7029089?type=season',
    ],
}

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ),
    'Referer': 'https://space.bilibili.com/19106800',
    'Accept': 'application/json, text/plain, */*',
}

REQUEST_INTERVAL_SEC = 0.8
MAX_RETRIES = 5
PAGE_SIZE = 30


def parse_list_url(url: str) -> tuple[int, int]:
    parsed = urllib.parse.urlparse(url)
    mid_match = re.search(r'^/(\d+)', parsed.path)
    if not mid_match:
        raise ValueError(f'无法从链接解析 mid: {url}')
    mid = int(mid_match.group(1))

    query = urllib.parse.parse_qs(parsed.query)
    season_id = query.get('season_id') or query.get('sid')
    if not season_id:
        list_match = re.search(r'/lists/(\d+)', parsed.path)
        if not list_match:
            raise ValueError(f'无法从链接解析合集 id: {url}')
        season_id = [list_match.group(1)]

    return mid, int(season_id[0])


def fetch_json(url: str) -> dict:
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=30) as resp:
                payload = json.loads(resp.read().decode('utf-8'))
            if payload.get('code') != 0:
                raise RuntimeError(f"接口返回错误: {payload.get('message', payload)}")
            time.sleep(REQUEST_INTERVAL_SEC)
            return payload['data']
        except (urllib.error.URLError, TimeoutError, RuntimeError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(REQUEST_INTERVAL_SEC * attempt)
                continue
            raise last_error from exc
    raise RuntimeError('请求失败')


def append_videos(videos: list[dict], seen_bvids: set[str], items: list[dict]) -> int:
    added = 0
    for item in items:
        bvid = item.get('bvid')
        title = item.get('title')
        if not bvid or not title or bvid in seen_bvids:
            continue
        seen_bvids.add(bvid)
        videos.append({
            '标题': title,
            '链接': f'https://www.bilibili.com/video/{bvid}',
        })
        added += 1
    return added


def fetch_collection_from_polymer(mid: int, season_id: int) -> tuple[str, list[dict]]:
    page_num = 1
    videos: list[dict] = []
    seen_bvids: set[str] = set()
    season_title = ''
    total = None

    while True:
        data = fetch_json(
            'https://api.bilibili.com/x/polymer/web-space/seasons_archives_list'
            f'?mid={mid}&season_id={season_id}&page_num={page_num}&page_size={PAGE_SIZE}'
        )
        if page_num == 1:
            meta = data.get('meta') or {}
            season_title = meta.get('title') or meta.get('name') or ''
            total = (data.get('page') or {}).get('total')

        archives = data.get('archives') or []
        if not archives:
            break

        append_videos(videos, seen_bvids, archives)

        if total is not None and len(videos) >= total:
            break
        if len(archives) < PAGE_SIZE:
            break
        page_num += 1

    return season_title, videos


def fetch_collection_from_fav(season_id: int) -> tuple[str, list[dict]]:
    page = 1
    videos: list[dict] = []
    seen_bvids: set[str] = set()
    season_title = ''

    while True:
        data = fetch_json(
            'https://api.bilibili.com/x/space/fav/season/list'
            f'?season_id={season_id}&pn={page}&ps={PAGE_SIZE}'
        )
        if page == 1:
            season_title = (data.get('info') or {}).get('title', '')

        medias = data.get('medias') or []
        if not medias:
            break

        added = append_videos(videos, seen_bvids, medias)
        if added == 0:
            break
        page += 1

    return season_title, videos


def fetch_collection(list_url: str) -> dict:
    mid, season_id = parse_list_url(list_url)
    errors: list[str] = []

    for fetcher in (fetch_collection_from_polymer, fetch_collection_from_fav):
        try:
            if fetcher is fetch_collection_from_polymer:
                season_title, videos = fetcher(mid, season_id)
            else:
                season_title, videos = fetcher(season_id)
            if videos:
                return {
                    '来源': list_url,
                    'mid': mid,
                    '合集ID': season_id,
                    '合集标题': season_title,
                    '视频': videos,
                }
        except RuntimeError as exc:
            errors.append(str(exc))

    detail = '；'.join(errors) if errors else '未获取到视频'
    raise RuntimeError(f'合集 {season_id} 抓取失败: {detail}')


result = {}
summary = {}
for category, urls in LIST_SOURCES.items():
    collections = [fetch_collection(url) for url in urls]
    result[category] = collections
    summary[category] = sum(len(item['视频']) for item in collections)

OUTPUT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'已写入 {OUTPUT_PATH.resolve()}')
for category, count in summary.items():
    print(f'  {category}: {count} 条视频，{len(result[category])} 个合集')
result

RuntimeError: 接口返回错误: 服务暂不可用